# CLMS CLCplus Europe

**Living Earth - Cube in a Box Demo Series**

---

*   **Objective:** Load and plot CLMS CLCplus LULUCF Instance land cover for Europe at 100 m.
*   **Products used:** [`clms_clcplus_europe_100m`](http://localhost/explorer/products/clms_clcplus_europe_100m)
*   **Source:** [Copernicus Land Monitoring Service / Dataset Catalog](https://browser.stac.dataspace.copernicus.eu/collections/clms_clcplus_lulucf-instance_europe_100m_yearly_v1)

---

## Background

The CORINE Land Cover Plus Land Use, Land-Use Change and Forestry Instance (CLCplus LULUCF Instance) is an annually updated, pan-European raster at 100 m resolution. It provides a spatially consistent proxy for land use reporting under the LULUCF regulation, derived from multiple Copernicus Land Monitoring Service (CLMS) high-resolution input datasets.

Each raster cell represents a dominant LULUCF land-use class. While each pixel corresponds primarily to one of six main LULUCF categories — forest land, grassland, cropland, settlements, wetlands, and other lands — the dataset further differentiates these into 27 subclasses. Reference years include 2018, 2021, 2022 and 2023, with annual updates from 2021 onward.

This product is not based directly on satellite image classification; it is produced by combining existing CLMS layers. It is Europe-only (EEA38 area).

## Description

This notebook demonstrates how to load and visualize the CLCplus Europe product, including inspecting measurements and using custom colormaps; and is similar to [ESRI_Land_Cover.ipynb](ESRI_Land_Cover.ipynb).

***

In [ ]:
import sys
sys.path.insert(1, './utils/')

In [ ]:
# reload module before executing code
%load_ext autoreload
%autoreload 2

import datacube
import time
import numpy as np
from utils.le_dc import get_product_bbox, get_patch_url
from utils.le_mapping import bbox_to_polygon, display_crosshair, MapHandler
from utils.le_plotting import plot_da_categories
from utils.le_tools import style_output_cells


### Connect to the datacube

In [ ]:
dc = datacube.Datacube(app='CLMS_CLCplus_Europe')


## Describe product measurements

In [ ]:
dc.list_measurements().loc['clms_clcplus_europe_100m']


## Load CLCplus Europe data from the datacube

As this product contains only a single integer band for a limited number of times, and consequently is way lighter than other available products, we will process it the light way, meaning without Daskerization (as in other products notebooks).

> **Note:** This product is indexed from Copernicus Data Space. CDSE S3 credentials must be configured by your administrator; when enabled, they are provided automatically in Jupyter. URL signing is handled by `get_patch_url`.


In [ ]:
# Check if default bbox is contained within the datacube
# and allow user to draw bbox if not.

product = 'clms_clcplus_europe_100m'

# configure a default bounding box (Europe — product is Europe-only) and visualize it
lat, lon = 50.85, 4.35
buffer = 0.1
default_bbox = (lon - buffer, lat - buffer, lon + buffer, lat + buffer)

product_bbox = get_product_bbox(dc, product, split_size=10, stability_threshold=4)

is_contained =(default_bbox[0] >= product_bbox[0] and
               default_bbox[1] >= product_bbox[1] and
               default_bbox[2] <= product_bbox[2] and
               default_bbox[3] <= product_bbox[3]
              )


In [ ]:
# Create an instance of MapHandler
map_handler = MapHandler()
m, drc = map_handler.create_map(vect=[bbox_to_polygon(default_bbox), bbox_to_polygon(product_bbox)],
                               draw_rect=True)
display(m)

# append crosshair
time.sleep(2)  # make sure m is fully displayed
display_crosshair()

In [ ]:
# Warn in case of full AoI
aoi_poly = map_handler.aoi_tupple

if aoi_poly is None:
    aoi_poly = tuple(default_bbox)
    if not is_contained:
        style_output_cells('salmon', border_color='red', border_width='2px')
        print('The area of interest polygon is located outside of the product extent.' + \
              '\nPlease draw a new area of interest in the previous cell.')
    else:
        # When is_contained is True and no polygon drawn - this is actually OK!
        style_output_cells()
        print('Default area of interest is contained within the product extent, but you can still draw another one in the previous cell.')
else:
    # A polygon was drawn
    style_output_cells()
    print('Custom area of interest polygon has been created.')

In [ ]:
# Create a query object (we don't need to define time range and measurements as it is a
# single-band product with a limited number of yearly layers) and use it to load the data

query = {
    'product': product,
    'x': (aoi_poly[0], aoi_poly[2]),
    'y': (aoi_poly[1], aoi_poly[3]),
    'output_crs': 'epsg:6933',
    'resolution': 100,
    'patch_url': get_patch_url(dc, product),
}

ds = dc.load(**query)
print(ds)


## Plot CLCplus Europe

To do so you need to first define the style of each Land Cover category


In [ ]:
# Plot the first time

# define Value, Color and Label triplets (CLCplus LULUCF classes)
vcls = [
    # Settlements (11–14)
    (11, (180, 60, 40), 's_burnt_areas'),
    (12, (230, 0, 0), 's_settlements'),
    (13, (255, 100, 100), 's_green_urban_areas'),
    (14, (200, 80, 80), 's_other_settlements'),
    # Forest land (21–25)
    (21, (80, 40, 20), 'fl_burnt_areas'),
    (22, (140, 180, 80), 'fl_transitional_woodland'),
    (23, (0, 140, 0), 'fl_deciduous'),
    (24, (0, 80, 0), 'fl_coniferous_evergreen'),
    (25, (40, 120, 40), 'fl_other_forestland'),
    # Cropland (31–34)
    (31, (160, 100, 40), 'cl_burnt_areas'),
    (32, (255, 220, 80), 'cl_annual_crops'),
    (33, (240, 180, 40), 'cl_perennial_crops'),
    (34, (220, 200, 100), 'cl_other_cropland'),
    # Grassland (41–45)
    (41, (140, 120, 40), 'gl_burnt_areas'),
    (42, (200, 230, 100), 'gl_pastures'),
    (43, (180, 160, 60), 'gl_shrubs'),
    (44, (220, 240, 140), 'gl_natural_grassland'),
    (45, (190, 210, 120), 'gl_other_grassland'),
    # Wetlands (51–55)
    (51, (0, 140, 160), 'wl_wetland_managed'),
    (52, (0, 180, 180), 'wl_wetland_unmanaged'),
    (53, (0, 100, 200), 'wl_water_managed'),
    (54, (0, 140, 220), 'wl_water_unmanaged'),
    (55, (60, 80, 100), 'wl_burnt_areas'),
    # Other land (61–64)
    (61, (180, 180, 180), 'ol_bare_soil_and_rocks'),
    (62, (240, 240, 250), 'ol_permanent_snow_and_ice'),
    (63, (200, 190, 140), 'ol_lichens_and_mosses'),
    (64, (160, 150, 130), 'ol_other_otherland'),
    # Special
    (254, (200, 200, 200), 'unclassified_clouds'),
    (255, (255, 255, 255), 'outside_area'),
]

da = ds["data"].isel(time=0)
plot_da_categories(da, vcls, title= da.time.values.astype('datetime64[Y]').astype(int) + 1970,
                   cb=True, figsize=(12, 6))


In [ ]:
# Plot all time, displaying a colorbar only midway

for idx in range(len(ds.time)):
    da = ds["data"].isel(time=idx)
    cb = idx == np.floor(len(ds.time) / 2)  # display colorbar midway
    plot_da_categories(da, vcls, title= da.time.values.astype('datetime64[Y]').astype(int) + 1970,
                       cb=cb, figsize=(12, 6))

***

## Additional information

**License:** The code in this notebook is slighly modified from https://github.com/digitalearthafrica/deafrica-sandbox-notebooks and licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0).

**Compatible datacube version:**

In [ ]:
print(datacube.__version__)

**Last tested:**

In [ ]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d')

In [ ]:
!pip freeze